# Cross-Corpus Generalization of Deep Learning Detectors for Fake News and Misinformation
## Multi-seed, three-corpus reproducible companion notebook (v2)

Runs end to end on a free Colab **T4 GPU** (Runtime -> Change runtime type -> T4 GPU),
no API keys, no manual downloads. Extends v1 with:

1. **Three corpora** (LIAR statements, McIntire articles, ISOT articles) and the full **3x3 cross-domain matrix**.
2. **Multi-seed** training for the stochastic models (BiLSTM, DistilBERT, RoBERTa) with **mean +/- standard deviation** reported.
3. **Cross-corpus de-duplication** to prevent train/test leakage between the two news corpora.
4. A **fixed per-corpus training budget** so corpus size is controlled across the comparison.
5. **Progressive saving**: `results/results.json` is rewritten after each model finishes, so a disconnect does not lose completed work.

> Expected runtime on a free T4 GPU: about **60-90 minutes** for 3 seeds. Set `SEEDS` below to control this.


## 0. Configuration and reproducibility

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" shap scikit-learn statsmodels matplotlib 2>/dev/null
import os, random, json, re, time, sys, hashlib
import numpy as np, torch

# ---------------- experiment configuration ----------------
SEEDS      = [42, 7, 123]      # add 2 more (e.g. [42,7,123,2024,31]) for a 5-seed run (~2x time)
DATA_SEED  = 42               # fixes splits/caps so the SAME data is used across model seeds
TRAIN_CAP  = 5000             # fixed training budget per corpus (controls corpus size)
TEST_CAP   = 1500             # cap on test size per corpus (keeps cross-domain eval fast)
TRANSFORMERS = ["distilbert-base-uncased", "roberta-base"]
EPOCHS     = 3
# ----------------------------------------------------------

os.environ["PYTHONHASHSEED"]="0"; random.seed(DATA_SEED); np.random.seed(DATA_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True); os.makedirs("results/figs", exist_ok=True)
import transformers, sklearn
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| sklearn", sklearn.__version__, "| device", DEVICE)
print("seeds:", SEEDS, "| train_cap:", TRAIN_CAP)
RESULTS = {"config": {"seeds": SEEDS, "data_seed": DATA_SEED, "train_cap": TRAIN_CAP,
                      "test_cap": TEST_CAP, "transformers": TRANSFORMERS, "epochs": EPOCHS}}
def save():   # progressive save
    with open("results/results.json","w") as f: json.dump(RESULTS, f, indent=2)


## 1. Data acquisition and harmonization

Three public corpora, harmonized to one binary label `y = 1` (fake/misinformation) vs `y = 0` (real).

* **LIAR** (Wang, 2017): short PolitiFact statements; six labels binarized {pants-fire, false, barely-true} -> fake.
* **McIntire**: full news articles labeled FAKE/REAL.
* **ISOT** (Ahmed, Traore & Saad, 2017): Reuters (real) vs flagged-site (fake) articles, via the GonzaloA mirror whose labels are documented as fake (0) / true (1).

None requires a social-media API. After loading, we remove any article that appears in more than one corpus (cross-corpus de-duplication) to prevent leakage.


In [ ]:
import pandas as pd, urllib.request, urllib.error
def clean(t):
    t=str(t).replace("\n"," ").replace("\r"," "); return re.sub(r"\s+"," ",t).strip()

# ---- LIAR (direct GitHub mirror, with retry) ----
UA={"User-Agent":"Mozilla/5.0 (reproducibility-notebook; academic research)"}
def fetch(url,path,retries=6):
    for i in range(retries):
        try:
            req=urllib.request.Request(url,headers=UA)
            with urllib.request.urlopen(req,timeout=90) as r, open(path,"wb") as f: f.write(r.read())
            if os.path.getsize(path)>0: return True
            raise IOError("empty")
        except Exception as e:
            if i<retries-1: time.sleep(5*(2**i)); continue
            print("  failed",os.path.basename(path),e); return False
os.makedirs("data",exist_ok=True)
liar_cols=['id','label','statement','subject','speaker','job','state','party',
           'barely_true','false','half_true','mostly_true','pants_fire','context']
FAKE_LIAR={'pants-fire','false','barely-true'}; REAL_LIAR={'half-true','mostly-true','true'}
liar_parts=[]
for sp in ['train','valid','test']:
    p=f"data/liar_{sp}.tsv"
    if not os.path.exists(p) or os.path.getsize(p)==0:
        fetch(f"https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/{sp}.tsv",p)
    df=pd.read_csv(p,sep='\t',header=None,names=liar_cols)
    df=df[df['label'].isin(FAKE_LIAR|REAL_LIAR)].copy()
    df['text']=df['statement'].map(clean); df['y']=df['label'].isin(FAKE_LIAR).astype(int)
    liar_parts.append(df[['text','y']])
# LIAR fallback via HF if the mirror is unavailable
if sum(len(x) for x in liar_parts)<1000:
    from datasets import load_dataset
    dd=load_dataset("liar"); lbl={0:'false',1:'half-true',2:'mostly-true',3:'true',4:'barely-true',5:'pants-fire'}
    liar_parts=[]
    for s in ['train','validation','test']:
        d=dd[s].to_pandas(); d['lab']=d['label'].map(lbl)
        d=d[d['lab'].isin(FAKE_LIAR|REAL_LIAR)]
        liar_parts.append(pd.DataFrame({'text':d['statement'].map(clean),'y':d['lab'].isin(FAKE_LIAR).astype(int)}))
liar_all=pd.concat(liar_parts,ignore_index=True)

# ---- McIntire (direct GitHub mirror) ----
mp="data/mcintire.csv"
if not os.path.exists(mp) or os.path.getsize(mp)==0:
    fetch("https://raw.githubusercontent.com/lutzhamel/fake-news/master/data/fake_or_real_news.csv",mp)
mc=pd.read_csv(mp)
mc_all=pd.DataFrame({'text':(mc['title'].fillna('')+'. '+mc['text'].fillna('')).map(clean),
                     'y':(mc['label'].str.upper()=='FAKE').astype(int)})

# ---- ISOT via GonzaloA HF mirror (label 1=true, 0=fake) ----
from datasets import load_dataset, concatenate_datasets
iso=load_dataset("GonzaloA/fake_news")
iso_df=pd.concat([iso[s].to_pandas() for s in iso.keys()],ignore_index=True)
# documented: label 1 = true/real, 0 = fake  ->  y_fake = 1 - label
iso_all=pd.DataFrame({'text':(iso_df['title'].fillna('')+'. '+iso_df['text'].fillna('')).map(clean),
                      'y':(1-iso_df['label'].astype(int))})

CORP={'LIAR':liar_all,'McIntire':mc_all,'ISOT':iso_all}
for k,v in CORP.items():
    v.drop_duplicates(subset='text',inplace=True)
    v=v[v['text'].str.len()>0].reset_index(drop=True); CORP[k]=v
    print(f"{k}: {len(v)} rows, fake_rate {v['y'].mean():.3f}, mean_chars {int(v['text'].str.len().mean())}")


In [ ]:
# Cross-corpus de-duplication: drop any text that appears in more than one corpus
def h(t): return hashlib.md5(re.sub(r'[^a-z0-9]','',t.lower())[:400].encode()).hexdigest()
seen={}
for name,df in CORP.items():
    for x in df['text']: seen.setdefault(h(x),set()).add(name)
dupe={k for k,v in seen.items() if len(v)>1}
for name in CORP:
    before=len(CORP[name]); CORP[name]=CORP[name][~CORP[name]['text'].map(lambda t:h(t) in dupe)].reset_index(drop=True)
    print(f"{name}: removed {before-len(CORP[name])} cross-corpus duplicates -> {len(CORP[name])}")


In [ ]:
# Stratified splits with fixed DATA_SEED; cap train/test for a controlled, fast comparison
from sklearn.model_selection import train_test_split
def split_cap(df):
    tr,te=train_test_split(df,test_size=0.2,random_state=DATA_SEED,stratify=df['y'])
    va,te=train_test_split(te,test_size=0.5,random_state=DATA_SEED,stratify=te['y'])
    if len(tr)>TRAIN_CAP: tr,_=train_test_split(tr,train_size=TRAIN_CAP,random_state=DATA_SEED,stratify=tr['y'])
    if len(te)>TEST_CAP:  te,_=train_test_split(te,train_size=TEST_CAP,random_state=DATA_SEED,stratify=te['y'])
    return tr.reset_index(drop=True),va.reset_index(drop=True),te.reset_index(drop=True)
DS={}; prof={}
for name,df in CORP.items():
    tr,va,te=split_cap(df); DS[name]={'train':tr,'valid':va,'test':te}
    prof[name]={'train_n':len(tr),'valid_n':len(va),'test_n':len(te),
                'train_fake_rate':round(float(tr['y'].mean()),4),
                'mean_chars_train':int(tr['text'].str.len().mean()),
                'median_chars_train':int(tr['text'].str.len().median())}
RESULTS['dataset_profile']=prof; save(); print(json.dumps(prof,indent=2))


## 2. Shared metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, f1_score
def evaluate(y,yp,ys):
    p,r,f,_=precision_recall_fscore_support(y,yp,average=None,labels=[0,1],zero_division=0)
    _,_,fm,_=precision_recall_fscore_support(y,yp,average='macro',zero_division=0)
    try: auc=roc_auc_score(y,ys)
    except Exception: auc=float('nan')
    return {'accuracy':round(accuracy_score(y,yp),4),'f1_macro':round(fm,4),
            'f1_fake':round(f[1],4),'roc_auc':round(float(auc),4)}
def ece(y,pr,bins=10):
    y=np.asarray(y);pr=np.asarray(pr);pred=(pr>=.5).astype(int);conf=np.maximum(pr,1-pr)
    cor=(pred==y).astype(float);ed=np.linspace(0,1,bins+1);e=0;N=len(y)
    for i in range(bins):
        m=(conf>ed[i])&(conf<=ed[i+1]) if i>0 else (conf>=ed[i])&(conf<=ed[i+1])
        if m.sum(): e+=(m.sum()/N)*abs(cor[m].mean()-conf[m].mean())
    return round(float(e),4)
def agg(list_of_metric_dicts):
    keys=list_of_metric_dicts[0].keys(); out={}
    for k in keys:
        v=[d[k] for d in list_of_metric_dicts]
        out[k+'_mean']=round(float(np.nanmean(v)),4); out[k+'_std']=round(float(np.nanstd(v)),4)
    return out


## 3. Classical baselines (deterministic): in-domain + 3x3 cross-domain

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
def make_vec(): return TfidfVectorizer(lowercase=True,stop_words='english',ngram_range=(1,2),
                                       min_df=2,max_df=0.9,sublinear_tf=True,max_features=50000)
def sc(m,X): return m.predict_proba(X)[:,1] if hasattr(m,'predict_proba') else \
    (lambda d:(d-d.min())/(d.max()-d.min()+1e-9))(m.decision_function(X))
MODELS={'LogReg':lambda:LogisticRegression(max_iter=2000,C=4.0,class_weight='balanced'),
        'LinSVM':lambda:LinearSVC(C=1.0,class_weight='balanced'),
        'MultiNB':lambda:MultinomialNB()}
fitted={}; RESULTS['classical_in_domain']={}
for dn,d in DS.items():
    vec=make_vec(); Xtr=vec.fit_transform(d['train']['text']); ytr=d['train']['y'].values
    Xte=vec.transform(d['test']['text']); yte=d['test']['y'].values
    RESULTS['classical_in_domain'][dn]={}
    for mn,mf in MODELS.items():
        clf=mf(); clf.fit(Xtr,ytr); RESULTS['classical_in_domain'][dn][mn]=evaluate(yte,clf.predict(Xte),sc(clf,Xte))
        fitted[(dn,mn)]=(vec,clf)
    print(dn,RESULTS['classical_in_domain'][dn]['LogReg'])
RESULTS['classical_cross_domain']={}
for s in DS:
    vec,clf=fitted[(s,'LogReg')]
    for t in DS:
        if s==t: continue
        Xte=vec.transform(DS[t]['test']['text']); yte=DS[t]['test']['y'].values
        RESULTS['classical_cross_domain'][f"{s}->{t}"]=evaluate(yte,clf.predict(Xte),sc(clf,Xte))
save(); print("cross-domain (LogReg):",json.dumps(RESULTS['classical_cross_domain'],indent=2))


## 4. BiLSTM (multi-seed): in-domain + 3x3 cross-domain

In [ ]:
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import torch.nn as nn
def tok(t): return re.findall(r"[a-z']+",t.lower())
class Vocab:
    def __init__(s,texts,mx=30000,mf=2):
        c=Counter(w for t in texts for w in tok(t)); s.itos=['<pad>','<unk>']+[w for w,f in c.most_common(mx) if f>=mf]
        s.stoi={w:i for i,w in enumerate(s.itos)}
    def enc(s,t,L): ids=[s.stoi.get(w,1) for w in tok(t)][:L]; return ids+[0]*(L-len(ids))
class TDS(Dataset):
    def __init__(s,df,v,L): s.x=[v.enc(t,L) for t in df['text']]; s.y=df['y'].tolist()
    def __len__(s): return len(s.y)
    def __getitem__(s,i): return torch.tensor(s.x[i]),torch.tensor(s.y[i])
class BiLSTM(nn.Module):
    def __init__(s,V,e=128,h=128):
        super().__init__()
        s.emb=nn.Embedding(V,e,padding_idx=0)
        s.lstm=nn.LSTM(e,h,batch_first=True,bidirectional=True)
        s.drop=nn.Dropout(0.3); s.fc=nn.Linear(2*h,2)
    def forward(s,x):
        o,_=s.lstm(s.emb(x)); return s.fc(s.drop(o.mean(1)))
MAXLEN={'LIAR':40,'McIntire':400,'ISOT':400}
def set_all(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
bilstm_seed_metrics={'in':{d:[] for d in DS}, 'cross':{f"{s}->{t}":[] for s in DS for t in DS if s!=t}}
for seed in SEEDS:
    set_all(seed); models={}
    for dn,d in DS.items():
        v=Vocab(d['train']['text']); L=MAXLEN[dn]
        tr=DataLoader(TDS(d['train'],v,L),batch_size=64,shuffle=True)
        m=BiLSTM(len(v.itos)).to(DEVICE); opt=torch.optim.Adam(m.parameters(),1e-3); lf=nn.CrossEntropyLoss()
        for _ in range(4):
            m.train()
            for xb,yb in tr: opt.zero_grad(); lf(m(xb.to(DEVICE)),yb.to(DEVICE)).backward(); opt.step()
        models[dn]=(m,v,L)
    def bpred(m,v,L,df):
        dl=DataLoader(TDS(df,v,L),batch_size=128); m.eval(); ys=[];yp=[];yt=[]
        with torch.no_grad():
            for xb,yb in dl:
                p=torch.softmax(m(xb.to(DEVICE)),1)[:,1].cpu().numpy(); ys+=p.tolist(); yp+=(p>=.5).astype(int).tolist(); yt+=yb.tolist()
        return evaluate(yt,yp,ys)
    for dn in DS:
        m,v,L=models[dn]; bilstm_seed_metrics['in'][dn].append(bpred(m,v,L,DS[dn]['test']))
    for s in DS:
        m,v,L=models[s]
        for t in DS:
            if s==t: continue
            bilstm_seed_metrics['cross'][f"{s}->{t}"].append(bpred(m,v,L,DS[t]['test']))
    print(f"BiLSTM seed {seed} done")
RESULTS['bilstm_in_domain']={d:agg(bilstm_seed_metrics['in'][d]) for d in DS}
RESULTS['bilstm_cross_domain']={k:agg(v) for k,v in bilstm_seed_metrics['cross'].items()}
save(); print("BiLSTM in-domain:",{d:RESULTS['bilstm_in_domain'][d]['accuracy_mean'] for d in DS})


## 5. Transformers (multi-seed): DistilBERT and RoBERTa

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
TMAX={'LIAR':64,'McIntire':256,'ISOT':256}
class HFDS(torch.utils.data.Dataset):
    def __init__(s,df,tk,L): s.enc=tk(list(df['text']),truncation=True,max_length=L,padding='max_length'); s.y=df['y'].tolist()
    def __len__(s): return len(s.y)
    def __getitem__(s,i): it={k:torch.tensor(v[i]) for k,v in s.enc.items()}; it['labels']=torch.tensor(s.y[i]); return it
def probs(tr,ds): import torch as T; return T.softmax(T.tensor(tr.predict(ds).predictions),1)[:,1].numpy()

RESULTS.setdefault('transformer_in_domain',{}); RESULTS.setdefault('transformer_cross_domain',{})
transformer_primary_probs={}   # (model,corpus)->(y,probs) at seed 42 for calibration
for mname in TRANSFORMERS:
    seed_in={d:[] for d in DS}; seed_cross={f"{s}->{t}":[] for s in DS for t in DS if s!=t}
    for seed in SEEDS:
        set_seed(seed); trainers={}
        for dn,d in DS.items():
            tk=AutoTokenizer.from_pretrained(mname)
            mdl=AutoModelForSequenceClassification.from_pretrained(mname,num_labels=2)
            L=TMAX[dn]
            args=TrainingArguments(output_dir=f"ckpt/{mname.split('/')[-1]}_{dn}_{seed}",
                num_train_epochs=EPOCHS,per_device_train_batch_size=16,per_device_eval_batch_size=64,
                learning_rate=2e-5,weight_decay=0.01,warmup_ratio=0.06,fp16=torch.cuda.is_available(),
                seed=seed,logging_steps=500,report_to=[],eval_strategy="no",save_strategy="no")
            tr=Trainer(model=mdl,args=args,train_dataset=HFDS(d['train'],tk,L)); tr.train()
            trainers[dn]=(tr,tk,L)
            pr=probs(tr,HFDS(d['test'],tk,L)); yte=d['test']['y'].values
            seed_in[dn].append(evaluate(yte,(pr>=.5).astype(int),pr))
            if seed==SEEDS[0]: transformer_primary_probs[(mname,dn)]=(yte,pr)
        for s in DS:
            tr,tk,_=trainers[s]
            for t in DS:
                if s==t: continue
                pr=probs(tr,HFDS(DS[t]['test'],tk,TMAX[s])); yte=DS[t]['test']['y'].values
                seed_cross[f"{s}->{t}"].append(evaluate(yte,(pr>=.5).astype(int),pr))
        print(f"{mname} seed {seed} done  ({time.strftime('%H:%M:%S')})")
        # progressive partial save after each seed
        RESULTS['transformer_in_domain'][mname]={d:agg(seed_in[d]) for d in DS if seed_in[d]}
        RESULTS['transformer_cross_domain'][mname]={k:agg(v) for k,v in seed_cross.items() if v}
        save()
    print(f"=== {mname} complete ===")


## 6. Calibration, interpretability, and statistics (primary seed)

In [ ]:
# Calibration ECE
RESULTS['calibration_ece']={}
for dn,d in DS.items():
    vec,lr=fitted[(dn,'LogReg')]; RESULTS['calibration_ece'][f"LogReg {dn} in-domain"]=ece(d['test']['y'].values,lr.predict_proba(vec.transform(d['test']['text']))[:,1])
for s in DS:
    for t in DS:
        if s==t: continue
        vec,lr=fitted[(s,'LogReg')]; RESULTS['calibration_ece'][f"LogReg {s}->{t}"]=ece(DS[t]['test']['y'].values,lr.predict_proba(vec.transform(DS[t]['test']['text']))[:,1])
pm=TRANSFORMERS[0]
for dn in DS:
    if (pm,dn) in transformer_primary_probs:
        y,pr=transformer_primary_probs[(pm,dn)]; RESULTS['calibration_ece'][f"{pm} {dn} in-domain"]=ece(y,pr)
# Interpretability: top fake-indicative tokens
RESULTS['top_features']={}
for dn in DS:
    vec,lr=fitted[(dn,'LogReg')]; names=np.array(vec.get_feature_names_out()); c=lr.coef_[0]
    RESULTS['top_features'][dn]={'fake_indicative':names[np.argsort(c)[-15:]][::-1].tolist()}
# Vocabulary overlap
from itertools import combinations
RESULTS['vocab_jaccard_top2000']={}
def tv(dn,k=2000): v,_=fitted[(dn,'LogReg')]; return set(v.get_feature_names_out()[:k])
for a,b in combinations(list(DS),2):
    RESULTS['vocab_jaccard_top2000'][f"{a}|{b}"]=round(len(tv(a)&tv(b))/len(tv(a)|tv(b)),4)
# Bootstrap CI + McNemar (LogReg vs LinSVM) per corpus
from statsmodels.stats.contingency_tables import mcnemar
def boot(y,yp,n=1000,seed=DATA_SEED):
    rng=np.random.default_rng(seed);y=np.asarray(y);yp=np.asarray(yp);N=len(y);v=[rng.integers(0,N,N) for _ in range(n)]
    vals=[f1_score(y[i],yp[i],pos_label=1,zero_division=0) for i in v]
    return [round(float(np.percentile(vals,2.5)),4),round(float(np.percentile(vals,97.5)),4)]
RESULTS['bootstrap_ci_f1fake']={}; RESULTS['mcnemar_logreg_vs_linsvm']={}
for dn,d in DS.items():
    vec,lr=fitted[(dn,'LogReg')]; _,sv=fitted[(dn,'LinSVM')]
    Xte=vec.transform(d['test']['text']); yte=d['test']['y'].values; lp=lr.predict(Xte); sp=sv.predict(Xte)
    RESULTS['bootstrap_ci_f1fake'][dn]=boot(yte,lp)
    tb=[[int(((lp==yte)&(sp==yte)).sum()),int(((lp==yte)&(sp!=yte)).sum())],
        [int(((lp!=yte)&(sp==yte)).sum()),int(((lp!=yte)&(sp!=yte)).sum())]]
    r=mcnemar(tb,exact=False,correction=True); RESULTS['mcnemar_logreg_vs_linsvm'][dn]={'pvalue':round(float(r.pvalue),6)}
save(); print("calibration:",RESULTS['calibration_ece']); print("jaccard:",RESULTS['vocab_jaccard_top2000'])


## 7. Summary (mean +/- std across seeds)

In [ ]:
def line(name,d): 
    return f"{name:<26} acc={d['accuracy_mean']:.3f}±{d['accuracy_std']:.3f}  F1m={d['f1_macro_mean']:.3f}  AUC={d['roc_auc_mean']:.3f}±{d['roc_auc_std']:.3f}"
print("="*70,"\nIN-DOMAIN (mean±std over",len(SEEDS),"seeds; classical deterministic)\n"+"="*70)
for dn in DS:
    print(f"\n[{dn}]")
    for m,r in RESULTS['classical_in_domain'][dn].items(): print(f"  {m:<12} acc={r['accuracy']:.3f}  F1m={r['f1_macro']:.3f}  AUC={r['roc_auc']:.3f}")
    print("  "+line('BiLSTM',RESULTS['bilstm_in_domain'][dn]))
    for m in TRANSFORMERS:
        if dn in RESULTS['transformer_in_domain'].get(m,{}): print("  "+line(m,RESULTS['transformer_in_domain'][m][dn]))
print("\n"+"="*70,"\nCROSS-DOMAIN 3x3 (RoBERTa mean±std shown; full matrix in results.json)\n"+"="*70)
for m in TRANSFORMERS:
    print(f"\n[{m}]")
    for k,r in RESULTS['transformer_cross_domain'].get(m,{}).items():
        print(f"  {k:<20} acc={r['accuracy_mean']:.3f}±{r['accuracy_std']:.3f}  AUC={r['roc_auc_mean']:.3f}")
print("\nSaved results/results.json — download and paste back to finalize the paper.")
